# Five common worry–harm clusters across World Risk Poll waves

We fit the 2021, 2023 and 2025 development populations independently with the same encoding. Each fit has seven weighted K-means centres, with the three centres for traffic, crime and severe weather mapped to the external-hazards cluster. The other four centres each define a cluster. This notebook replays the saved five-cluster evidence. The input packages contain aggregated answer-pattern counts and survey mass, fitted centres and predictions for those patterns. They contain no underlying respondent records. Run All checks assignments, resampling evidence, common rules and trends. An optional cell repeats the exhaustive rule search. No clustering fits are launched here.

In [1]:
from pathlib import Path
from itertools import product, combinations
import hashlib
import json
import tempfile

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.metrics import adjusted_rand_score
from threadpoolctl import threadpool_limits
from IPython.display import display

HERE=next((p for q in [Path.cwd(),*Path.cwd().parents] for p in [q,q/'paper',q/'github_publication/paper']
           if (p/'assets/shared_five_manifest.json').is_file()))
ASSETS=HERE/'assets'
manifest=json.loads((ASSETS/'shared_five_manifest.json').read_text())
for filename,expected in manifest['files_sha256'].items():
    assert hashlib.sha256((ASSETS/filename).read_bytes()).hexdigest()==expected

report=json.loads((ASSETS/'shared_five_report.json').read_text())
conditions=json.loads((ASSETS/'shared_five_rule_conditions.json').read_text())
RULE_NAMES=report['names']
YEARS=[2021,2023,2025]
with np.load(ASSETS/'shared_five_evidence.npz',allow_pickle=False) as pack:
    evidence={key:pack[key] for key in pack.files}
assert evidence['reference_counts'].sum()==90395

def geometry(answers):
    x=np.where(answers[:,:6]==2,1.,np.where(answers[:,:6]==1,1/3,0.))
    mean=x.mean(1,keepdims=True)
    return np.column_stack([(x-mean)*np.sqrt(.625/6),mean*np.sqrt(.625/6),
        answers[:,6:]*np.sqrt(.375/14),np.zeros(len(x))])

def predict(answers,year):
    centres=evidence[f'{year}_centres']
    groups=evidence[f'{year}_centre_to_cluster']
    return groups[cdist(geometry(answers),centres).argmin(1)]

def tab(x,y,weights):
    return np.bincount(5*x.astype('int64')+y,weights=weights,minlength=25).reshape(5,5)

def ari(t):
    def n2(v):return np.sum(v*(v-1)/2)
    n=t.sum()
    x,y=n2(t.sum(0)),n2(t.sum(1))
    e=x*y/(n*(n-1)/2)
    return (n2(t)-e)/((x+y)/2-e) if (x+y)/2!=e else 1.

reference=evidence['reference_answers']
counts=evidence['reference_counts']
mass=evidence['reference_mass']
for year in YEARS:
    for block in ['reference', 'native']:
        prefix='reference' if block=='reference' else f'native_{year}'
        key=f'reference_{year}_labels' if block=='reference' else f'native_{year}_labels'
        np.testing.assert_array_equal(predict(evidence[prefix+'_answers'],year),evidence[key])
print('Verified every saved point prediction against the fitted centres and group map.')

Verified every saved point prediction against the fitted centres and group map.


## Agreement on the same 2025 respondents

The counts give ordinary adjusted Rand index (ARI). Relative weights within a country follow WGT and every country has equal total weight. For a cluster, weighted Jaccard divides the weight of the intersection by that of the union under the two fits. Saved refit predictions are already aligned via their own wave's point fit to the 2025 labels. Use all 2,500 pairs for each wave pair and type of refit.

In [2]:
point_rows=[]
distributions=[]
cluster_rows=[]
for i,j in combinations(YEARS,2):
    x,y=[evidence[f'reference_{wave}_labels'] for wave in [i,j]]
    a=ari(tab(x,y,counts))
    agreement=np.trace(tab(x,y,mass))
    assert abs(a-adjusted_rand_score(np.repeat(x,counts),np.repeat(y,counts)))<1e-12
    point_rows.append(dict(year_a=i,year_b=j,ARI=a,weighted_agreement=agreement))
    for test in ['respondents','countries']:
        left=evidence[f'{i}_{test}_predictions']
        right=evidence[f'{j}_{test}_predictions']
        aris,jaccards=[],[]
        for x,y in product(left,right):
            aris.append(ari(tab(x,y,counts)))
            w=tab(x,y,mass)
            hit=np.diag(w)
            jaccards.append(hit/(w.sum(0)+w.sum(1)-hit))
        assert len(aris)==2500
        distributions.append(dict(year_a=i,year_b=j,test=test,p10=np.quantile(aris,.1),median=np.median(aris)))
        cluster_rows.extend(dict(year_a=i,year_b=j,test=test,cluster=c,p10=v)
                            for c,v in enumerate(np.quantile(jaccards,.1,axis=0)))

cross=pd.DataFrame(distributions)
jaccard=pd.DataFrame(cluster_rows).groupby('cluster').p10.min()
np.testing.assert_allclose(jaccard,report['recovery'],atol=1e-12)
np.testing.assert_allclose(cross[['p10','median']],
    pd.DataFrame(report['crossARI']).set_index(['year_a','year_b','test'])
    .loc[list(zip(cross.year_a,cross.year_b,cross.test)),['p10','median']],atol=1e-12)
display(pd.DataFrame(point_rows).round(4))
display(cross.round(4))
display(pd.DataFrame({'Cluster':RULE_NAMES,'Lowest cross-wave Jaccard p10':jaccard}).round(4))

,year_a,year_b,ARI,weighted_agreement
0,2021,2023,0.9652,0.9832
1,2021,2025,0.9371,0.9707
2,2023,2025,0.9374,0.9707


,year_a,year_b,test,p10,median
0,2021,2023,respondents,0.9129,0.9608
1,2021,2023,countries,0.9038,0.9350
2,2021,2025,respondents,0.9308,0.9409
3,2021,2025,countries,0.9219,0.9404
4,2023,2025,respondents,0.9295,0.9369
5,2023,2025,countries,0.9196,0.9273


,Cluster,Lowest cross-wave Jaccard p10
cluster,,
0,External hazards,0.8966
1,Basic necessities,0.8432
2,Mental-health worry,0.8560
3,Harm to people they know,0.8628
4,Few very high worries,0.9717


## One common rule for each cluster

The rule language uses AND and OR with at most three condition occurrences. Precision is the weighted share of rule matches in the target cluster. Coverage is the weighted share of cluster members matching its rule. Every selected rule reaches 80% precision in each wave. Among rules within one percentage point of maximum 2025 coverage, choose lowest cost, then highest 2025 precision and coverage, fewest original questions and canonical formula order.

The known-person-harm formula can show exactly four domains in its first branch because its other branch covers five or more. The few-very-high-worries rule allows food to be rated at any level and requires the other five domains to be below very worried. Its cost is three in the original search language.

In [3]:
def condition_masks(answers, conditions):
    counts = {'known': answers[:, 6:13].sum(1), 'personal': answers[:, 13:20].sum(1),
              'very': (answers[:, :6] == 2).sum(1), 'not': (answers[:, :6] == 0).sum(1),
              'some': (answers[:, :6] >= 1).sum(1)}
    operations = {'eq': np.equal, 'ge': np.greater_equal, 'le': np.less_equal}
    return np.column_stack([
        operations[c['op']](answers[:, c['j']] if 'j' in c else counts[c['base']], c['value'])
        for c in conditions])

def rule_clauses(kind, i, j=-1, k=-1):
    i, j, k = int(i), int(j), int(k)
    clauses = {0: [(i,)], 1: [(i, j)], 2: [(i,), (j,)],
               3: [(i, j, k)], 4: [(i,), (j,), (k,)],
               5: [(i,), (j, k)], 6: [(j,), (i, k)], 7: [(k,), (i, j)]}[int(kind)]
    return tuple(sorted(tuple(sorted(c)) for c in clauses))

def formula_string(clauses):
    return json.dumps({'clauses': clauses}, separators=(',', ':'))

def readable_rule(clauses, conditions):
    symbols = {'known': 'K', 'personal': 'P', 'very': 'V', 'not': 'N', 'some': 'A'}
    def atom(c):
        if c['kind'] == 'count_test':
            return symbols[c['base']] + (' >= ' if c['op'] == 'ge' else ' <= ') + str(c['value'])
        return c['label']
    parts = [' AND '.join(atom(conditions[i]) for i in clause) for clause in clauses]
    return ' OR '.join('(' + text + ')' if len(clauses)>1 and len(clause)>1 else text
                       for clause, text in zip(clauses, parts))

def formula_mask(clauses, tests):
    result = np.zeros(len(tests), dtype=bool)
    for clause in clauses:
        result |= np.all(tests[:, list(clause)], axis=1)
    return result

def predicate_intersections(population, assignment, n_clusters):
    tests, weights = population['tests'], population['weights']
    pattern_labels = np.full(len(weights), -1, dtype=int)
    pattern_labels[population['inverse']] = assignment
    assert np.array_equal(pattern_labels[population['inverse']], assignment)
    n = tests.shape[1]
    triples = np.array(list(combinations(range(n), 3)), dtype=np.int16)
    one = np.zeros((n, n_clusters))
    two = np.zeros((n, n, n_clusters))
    three = np.zeros((len(triples), n_clusters))
    with threadpool_limits(limits=4):
        for cluster in range(n_clusters):
            mask = pattern_labels == cluster
            block = tests[mask].astype(float)
            one[:, cluster] = block.T @ weights[mask]
            two[:, :, cluster] = block.T @ (block * weights[mask, None])
        offset = 0
        for i in range(n-2):
            j, k = np.triu_indices(n-i-1, k=1)
            for cluster in range(n_clusters):
                mask = tests[:, i] & (pattern_labels == cluster)
                block = tests[mask, i+1:].astype(float)
                joint = block.T @ (block * weights[mask, None])
                three[offset:offset+len(j), cluster] = joint[j, k]
            offset += len(j)
    assert offset == len(triples)
    return one, two, three, triples, np.bincount(pattern_labels, weights=weights, minlength=n_clusters)

def extract_description_rules(population, assignment, model, conditions,
                              candidate_file=None, epsilon=0.01, precision_floor=0.80):
    """Exhaustive DNF search, expanded cost <=3, with the agreed deterministic selection."""
    n = len(conditions)
    one, two, three, triples, totals = predicate_intersections(population, assignment, model['k'])
    pairs = np.array(list(combinations(range(n), 2)), dtype=np.int16)
    question_bits = np.array([sum(1 << q for q in set(c['questions'])) for c in conditions])
    bit_counts = np.array([i.bit_count() for i in range(1 << 13)])
    archives = []
    evaluated = 0

    def consider(kind, ids, intersections, cost):
        nonlocal evaluated
        # Roundoff in inclusion-exclusion can produce tiny values for empty events.
        intersections = np.where(np.abs(intersections)<1e-14, 0., intersections)
        intersections = np.minimum(np.maximum(intersections, 0.), totals)
        mass = intersections.sum(1)
        precision = np.divide(intersections, mass[:, None], out=np.zeros_like(intersections),
                              where=mass[:, None]>0)
        coverage = intersections / totals
        row, cluster = np.nonzero((mass[:, None]>0) & (precision>=precision_floor-1e-12))
        evaluated += len(ids)
        if not len(row):
            return
        selected = ids[row]
        q = bit_counts[np.bitwise_or.reduce(question_bits[selected], axis=1)]
        clauses = [rule_clauses(kind, *v) for v in selected]
        archives.append(pd.DataFrame(dict(cluster=cluster, kind=kind, i=selected[:,0],
            j=selected[:,1] if selected.shape[1]>1 else -1,
            k_atom=selected[:,2] if selected.shape[1]>2 else -1, cost=cost, questions=q,
            precision=precision[row,cluster], coverage=coverage[row,cluster],
            selected_mass=mass[row], true_mass=intersections[row,cluster],
            canonical_formula=[formula_string(c) for c in clauses])))

    consider(0, np.arange(n).reshape(-1,1), one.copy(), 1)
    a,b = pairs.T
    consider(1, pairs, two[a,b].copy(), 2)
    consider(2, pairs, one[a]+one[b]-two[a,b], 2)
    i,j,k = triples.T
    consider(3, triples, three.copy(), 3)
    consider(4, triples, one[i]+one[j]+one[k]-two[i,j]-two[i,k]-two[j,k]+three, 3)
    consider(5, triples, one[i]+two[j,k]-three, 3)
    consider(6, triples, one[j]+two[i,k]-three, 3)
    consider(7, triples, one[k]+two[i,j]-three, 3)
    assert evaluated == 1491084
    candidates = pd.concat(archives, ignore_index=True) if archives else pd.DataFrame(columns=['cluster'])
    if candidate_file is not None:
        candidates.to_csv(candidate_file, index=False, compression='gzip')
    selected_rules = []
    for cluster in range(model['k']):
        group = candidates.loc[candidates.cluster.eq(cluster)].copy()
        common = dict(wave=model['year'], trial=model['trial'], model_id=model['model_id'],
                      n_clusters=model['k'], cluster=cluster, formulas_evaluated=evaluated,
                      admissible_candidates=len(group), epsilon=epsilon, precision_floor=precision_floor)
        if group.empty:
            # A missing compact description does not remove an otherwise passing clustering.
            selected_rules.append(dict(common, status='no admissible rule under the declared budget'))
            continue
        maximum = group.coverage.max()
        band = group.loc[group.coverage>=maximum-epsilon-1e-12].copy()
        band['_p'] = band.precision.round(12)
        band['_r'] = band.coverage.round(12)
        best = band.sort_values(['cost','_p','_r','questions','canonical_formula'],
                               ascending=[True,False,False,True,True]).iloc[0]
        record = best.drop(labels=['_p','_r']).to_dict()
        clauses = rule_clauses(record['kind'],record['i'],record['j'],record['k_atom'])
        selected_rules.append(dict(record, **{key:value for key,value in common.items() if key!='cluster'},
                                   status='selected', best_coverage=maximum,
                                   rule=readable_rule(clauses, conditions)))
    return pd.DataFrame(selected_rules)

In [4]:
def rule_answers(model_answers):
    # Model inputs are worry, personal harm, known-person harm.
    # The original predicate IDs instead put known-person harm first.
    return model_answers[:,[*range(6),*range(13,20),*range(6,13)]]

def rule_measure(mask,target,mass,cluster):
    member=target==cluster
    hit=mass[mask&member].sum()
    selected=mass[mask].sum()
    total=mass[member].sum()
    return dict(precision=hit/selected if selected else np.nan,coverage=hit/total,
                selected_mass=selected,cluster_mass=total,intersection_mass=hit)

def score_rule_catalogue(candidates,answers,labels,mass,conditions):
    # Scores for every candidate that cleared 80% in 2025. The intersections
    # use the same inclusion–exclusion formulas as the full enumeration.
    population=dict(tests=condition_masks(rule_answers(answers),conditions),
                    weights=mass,inverse=np.arange(len(answers)))
    one,two,three,triples,totals=predicate_intersections(population,labels,5)
    n=len(conditions)
    lookup=np.full((n,n,n),-1,dtype='int32')
    lookup[tuple(triples.T)]=np.arange(len(triples))
    values=np.empty((len(candidates),2))
    for family,block in candidates.groupby('kind'):
        i,j,k=block[['i','j','k_atom']].to_numpy().T
        if family==0:
            intersections=one[i]
        elif family==1:
            intersections=two[i,j]
        elif family==2:
            intersections=one[i]+one[j]-two[i,j]
        else:
            positions=lookup[i,j,k]
            assert np.min(positions)>=0
            joint=three[positions]
            if family==3:
                intersections=joint
            elif family==4:
                intersections=one[i]+one[j]+one[k]-two[i,j]-two[i,k]-two[j,k]+joint
            elif family==5:
                intersections=one[i]+two[j,k]-joint
            elif family==6:
                intersections=one[j]+two[i,k]-joint
            else:
                assert family==7
                intersections=one[k]+two[i,j]-joint
        intersections=np.clip(np.where(abs(intersections)<1e-14,0.,intersections),0,totals)
        predicted_mass=intersections.sum(1)
        group=block.cluster.to_numpy()
        true=intersections[np.arange(len(block)),group]
        precision=np.divide(true,predicted_mass,out=np.zeros(len(block)),where=predicted_mass>0)
        values[block.index]=np.column_stack([precision,true/totals[group]])
    return values

def select_joint_rules(candidates,qualifying,conditions,criterion):
    selected=[]
    for c in range(5):
        pool=candidates.loc[qualifying & candidates.cluster.eq(c)]
        if pool.empty:
            selected.append(dict(criterion=criterion,cluster=c,name=RULE_NAMES[c],status='no admissible rule'))
            continue
        maximum=pool.coverage_2025.max()
        band=pool.loc[pool.coverage_2025.ge(maximum-.01-1e-12)].copy()
        band['_p']=band.precision_2025.round(12)
        band['_r']=band.coverage_2025.round(12)
        best=band.sort_values(['cost','_p','_r','questions','canonical_formula'],
                              ascending=[True,False,False,True,True]).iloc[0]
        formula=tuple(tuple(v) for v in json.loads(best.canonical_formula)['clauses'])
        selected.append(dict(criterion=criterion,cluster=c,name=RULE_NAMES[c],status='selected',
            canonical_formula=best.canonical_formula,rule=readable_rule(formula,conditions),
            cost=int(best.cost),questions=int(best.questions),eligible_candidates=len(pool),
            candidates_in_epsilon_band=len(band),best_coverage_2025=maximum))
    return pd.DataFrame(selected)

In [5]:
rows=[]
for y in YEARS:
    x=evidence[f'native_{y}_answers']
    labels=predict(x,y)
    w=evidence[f'native_{y}_mass']
    tests=condition_masks(rule_answers(x),conditions)
    for c,cl in enumerate(report['models'][f'full_{y}']['clusters']):
        rule=cl['rule']
        m=formula_mask(rule['canonical']['clauses'],tests)
        quality=rule_measure(m,labels,w,c)
        np.testing.assert_allclose([quality['precision'],quality['coverage']],
                                   [rule['precision'],rule['coverage']],atol=2e-12)
        assert quality['precision']>=.8 and rule['cost']<=3
        rows.append(dict(wave=y,cluster=c,name=cl['name'],rule=rule['text'],cost=rule['cost'],**quality))
results=pd.DataFrame(rows)
display(results[['wave','name','precision','coverage','cost']].round(4))

RUN_EXHAUSTIVE_SEARCH=False
if RUN_EXHAUSTIVE_SEARCH:
    x=evidence['native_2025_answers']
    pop=dict(tests=condition_masks(rule_answers(x), conditions),weights=evidence['native_2025_mass'],
             inverse=np.arange(len(x)))
    with tempfile.TemporaryDirectory() as t:
        path=Path(t)/'admissible.csv.gz'
        extract_description_rules(pop,predict(x,2025),dict(k=5,year=2025,trial='five',model_id='2025:five'),
                                  conditions,candidate_file=path)
        candidates=pd.read_csv(path,float_precision='round_trip').rename(columns={
            'precision':'precision_2025','coverage':'coverage_2025'})
    for y in [2021,2023]:
        x=evidence[f'native_{y}_answers']
        candidates[[f'precision_{y}',f'coverage_{y}']]=score_rule_catalogue(
            candidates,x,predict(x,y),evidence[f'native_{y}_mass'],conditions)
    good=candidates[[f'precision_{y}' for y in YEARS]].ge(.8-1e-12).all(axis=1)
    selected=select_joint_rules(candidates,good,conditions,'native')
    for r in selected.itertuples():
        assert json.loads(r.canonical_formula)==report['models']['full_2025']['clusters'][r.cluster]['rule']['canonical']
    print('All five common rules were reproduced by exhaustive search.')

,wave,name,precision,coverage,cost
0,2021,External hazards,0.8825,0.9096,3
1,2021,Basic necessities,0.8110,0.8536,3
2,2021,Mental-health worry,0.8612,0.9250,3
3,2021,Harm to people they know,0.8384,0.7410,3
4,2021,Few very high worries,0.9333,0.9874,3
5,2023,External hazards,0.8806,0.9017,3
6,2023,Basic necessities,0.8198,0.8363,3
7,2023,Mental-health worry,0.8159,0.9176,3
8,2023,Harm to people they know,0.8106,0.7340,3
9,2023,Few very high worries,0.9131,0.9813,3


## Changes in frequency using the same 2025 definitions

Apply the fixed 2025 model to each wave's own complete respondents. Average country shares across the same 92 development countries, and draw 1,000 paired country bootstrap samples for the change. This keeps the definition and country frame fixed as the survey wave changes.

In [6]:
shares=evidence['fixed_country_shares']
mean=shares.mean(axis=1)
np.testing.assert_allclose(mean,[report['shares'][str(y)] for y in YEARS],atol=1e-12)
display(pd.DataFrame(mean.T,index=RULE_NAMES,columns=YEARS).mul(100).round(2))
rng=np.random.default_rng(20261118)
indices=rng.integers(0,92,(1000,92))
changes=shares[2]-shares[0]
bounds=np.quantile(changes[indices].mean(1),[.025,.975],axis=0)
np.testing.assert_allclose(changes.mean(0),[r['delta'] for r in report['fixedChanges']],atol=1e-12)
np.testing.assert_allclose(bounds,[[r[k] for r in report['fixedChanges']] for k in ['lower','upper']],atol=1e-12)
display(pd.DataFrame({'Change':changes.mean(0),'Lower':bounds[0],'Upper':bounds[1]},index=RULE_NAMES).mul(100).round(2))

,2021,2023,2025
External hazards,28.65,30.02,30.44
Basic necessities,13.34,12.86,12.19
Mental-health worry,13.40,13.58,11.56
Harm to people they know,8.74,8.82,8.90
Few very high worries,35.87,34.71,36.92


,Change,Lower,Upper
External hazards,1.79,0.17,3.35
Basic necessities,-1.16,-2.07,-0.10
Mental-health worry,-1.83,-2.84,-0.91
Harm to people they know,0.15,-1.06,1.45
Few very high worries,1.05,-1.26,3.15


## Influence of encoded coordinates and original questions

We inspect the current frozen models on their native development populations. No centres or merge maps are fitted again. For each respondent, the mean is the average of that respondent's encoded worry answers. A relative worry coordinate is one answer minus this mean. Thus “water relative to mean” measures how much water worry stands above or below that person's other worries.

Three calculations answer different questions. For encoded coordinate $j$, weighted separation is $B_j=\sum_c W_c(\bar z_{cj}-\bar z_j)^2$; its share is $B_j/\sum_\ell B_\ell$. The feature scaling is the scaling used in the model. Coordinate omission drops the chosen squared-distance term for both respondents and centres, leaving all other terms fixed, and counts final-cluster changes. For original-question influence, replace an answer with an independent donor value from the same wave's weighted marginal, re-encode the complete answer vector, and count assignment changes. Enumerating every possible donor value gives the expectation exactly. Joint harm answers preserve the personal/known-person pair.

The mean and relative coordinates are derived from the same answers. Their omission sensitivities overlap and are not causal or uniquely attributable effects. The question calculation changes all affected derived coordinates coherently. These are the checks in Supplementary Section S2; the supplied manifest verifies the aggregate inputs and expected results.

In [7]:
from scipy.optimize import linear_sum_assignment
FI_DOMAINS=['food','water','crime','severe weather','traffic','mental health']
FI_WDOMAINS=['food','water','crime','severe weather','mental health']
FI_HARM=['food','water','crime','severe weather','traffic','mental health','work']
def fi_full_geometry(answers):
    v = np.array([0.0, 1 / 3, 1.0])[answers[:, :6]]
    m = v.mean(1, keepdims=True)
    return np.column_stack([(v - m) * np.sqrt(0.625 / 6), m * np.sqrt(0.625 / 6), answers[:, 6:] * np.sqrt(0.375 / 14), np.zeros(len(v))])

def fi_worry_geometry(answers, lam):
    v = np.array([0.0, 0.75, 1.0])[answers]
    m = v.mean(1, keepdims=True)
    return np.column_stack([v - m, np.sqrt(lam) * m]) / np.sqrt(5)

def fi_match_map(a, b, w, k):
    t = np.bincount(k * a.astype(int) + b, weights=w, minlength=k * k).reshape(k, k)
    i, j = linear_sum_assignment(-t)
    m = np.empty(k, dtype=np.int8)
    m[j] = i
    return m

def fi_means(X, labels, w, k):
    return np.asarray([np.average(X[labels == c], weights=w[labels == c], axis=0) for c in range(k)])

def fi_predict_model(model, answers=None, omit_parent=(), omit_child=(), mean_lambda=None):
    a = model['answers'] if answers is None else answers
    if model['family'] == 'full':
        X = fi_full_geometry(a)
        C = model['centres'].copy()
        if mean_lambda is not None:
            X[:, 6] *= np.sqrt(mean_lambda)
            C[:, 6] *= np.sqrt(mean_lambda)
        keep = np.setdiff1d(np.arange(X.shape[1]), omit_parent)
        return model['groups'][cdist(X[:, keep], C[:, keep], 'sqeuclidean').argmin(1)]
    X = fi_worry_geometry(a, 1.5)
    C = model['centres'].copy()
    if mean_lambda is not None:
        scale = np.sqrt(mean_lambda / 1.5)
        X[:, 5] *= scale
        C[:, 5] *= scale
    keep = np.setdiff1d(np.arange(X.shape[1]), omit_parent)
    parent = cdist(X[:, keep], C[:, keep], 'sqeuclidean').argmin(1)
    cut = parent == 2
    Z = fi_worry_geometry(a, 0.0)
    C2 = model['child_centres']
    keep2 = np.setdiff1d(np.arange(Z.shape[1]), omit_child)
    child = cdist(Z[:, keep2], C2[:, keep2], 'sqeuclidean').argmin(1)
    p = parent.astype(np.int8)
    p[cut] = 2 + child[cut]
    return model['groups'][p]

def fi_decomposition(X, labels, w, k, names, meta):
    w = w / w.sum()
    mu = fi_means(X, labels, w, k)
    overall = np.average(X, weights=w, axis=0)
    wc = np.bincount(labels, weights=w, minlength=k)
    by_cluster = wc[:, None] * (mu - overall) ** 2
    B = by_cluster.sum(0)
    T = (w[:, None] * (X - overall) ** 2).sum(0)
    residual = (w[:, None] * (X - mu[labels]) ** 2).sum(0)
    np.testing.assert_allclose(T, B + residual, atol=3e-15, rtol=3e-13)
    share = B / B.sum()
    eta = np.divide(B, T, out=np.zeros_like(B), where=T > 1e-25)
    rows = [dict(**meta, feature=name, TSS=float(T[j]), BSS=float(B[j]), WSS=float(residual[j]), separation_share=float(share[j]), eta_squared=float(eta[j])) for j, name in enumerate(names)]
    clusters = [dict(**meta, cluster=c, feature=name, BSS=float(by_cluster[c, j]), within_cluster_separation_share=float(by_cluster[c, j] / by_cluster[c].sum())) for c in range(k) for j, name in enumerate(names)]
    return (rows, clusters)

def fi_omission_list(family):
    if family == 'full':
        rows = [(t + ' (relative worry)', [j], [], 'coordinate') for j, t in enumerate(FI_DOMAINS)]
        rows += [('Overall worry mean', [6], [], 'coordinate')]
        rows += [('Personal harm: ' + t, [7 + j], [], 'coordinate') for j, t in enumerate(FI_HARM)]
        rows += [('Known-person harm: ' + t, [14 + j], [], 'coordinate') for j, t in enumerate(FI_HARM)]
        rows += [('All relative worries', list(range(6)), [], 'block'), ('All worry coordinates', list(range(7)), [], 'block'), ('All personal harm', list(range(7, 14)), [], 'block'), ('All known-person harm', list(range(14, 21)), [], 'block'), ('All harm', list(range(7, 21)), [], 'block')]
    else:
        rows = [(t + ' (relative worry, both stages)', [j], [j], 'coordinate') for j, t in enumerate(FI_WDOMAINS)]
        rows += [('Overall worry mean', [5], [5], 'coordinate'), ('All relative worries', list(range(5)), list(range(5)), 'block')]
        rows += [(t + ' (parent only)', [j], [], 'stage') for j, t in enumerate(FI_WDOMAINS)]
        rows += [(t + ' (child only)', [], [j], 'stage') for j, t in enumerate(FI_WDOMAINS)]
    return rows

def fi_randomisation_units(family):
    if family == 'worry':
        return [(t + ' worry', [j], 'original worry question') for j, t in enumerate(FI_WDOMAINS)]
    rows = [(t + ' worry', [j], 'original worry question') for j, t in enumerate(FI_DOMAINS)]
    rows += [('Personal harm: ' + t, [6 + j], 'personal indicator') for j, t in enumerate(FI_HARM)]
    rows += [('Known-person harm: ' + t, [13 + j], 'known-person indicator') for j, t in enumerate(FI_HARM)]
    rows += [('Harm question: ' + t, [6 + j, 13 + j], 'joint personal and known-person') for j, t in enumerate(FI_HARM)]
    return rows

def fi_inspect(models):
    pieces={name:[] for name in ['separation','cluster_separation','omissions','omission_clusters','questions','question_clusters']}
    for (family,y),m in models.items():
        X=m['X'];w=m['weights'];a=m['answers'];labels=m['labels'];k=m['k']
        np.testing.assert_array_equal(fi_predict_model(m),labels)
        if family=='full':
            names=[v+' (relative worry)' for v in FI_DOMAINS]+['Overall worry mean']
            names+=['Personal harm: '+v for v in FI_HARM]+['Known-person harm: '+v for v in FI_HARM]+['Unused zero coordinate']
            stages=[('final five clusters',X,labels,w,5,names)]
        else:
            names=[v+' (relative worry)' for v in FI_WDOMAINS]+['Overall worry mean']
            parent=m['parent_labels'];cut=parent==2
            child=cdist(m['child_X'],m['child_centres'],'sqeuclidean').argmin(1)
            stages=[('parent stage',X,parent,w,3,names),('child stage',m['child_X'][cut],child[cut],w[cut],2,names)]
        for stage,z,l,ww,n,names in stages:
            rows,clusters=fi_decomposition(z,l,ww,n,names,dict(family=family,wave=y,stage=stage))
            pieces['separation']+=rows;pieces['cluster_separation']+=clusters
        for name,parent,child,kind in fi_omission_list(family):
            pred=fi_predict_model(m,omit_parent=parent,omit_child=child)
            meta=dict(family=family,wave=y,feature=name,kind=kind)
            pieces['omissions'].append(dict(**meta,changed=float(w[pred!=labels].sum()/w.sum())))
            for c in range(k):
                old=labels==c;new=pred==c;hit=w[old&new].sum()
                pieces['omission_clusters'].append(dict(**meta,cluster=c,recovery=float(hit/w[old].sum()),Jaccard=float(hit/w[old|new].sum())))
        for name,cols,kind in fi_randomisation_units(family):
            states,inv=np.unique(a[:,cols],axis=0,return_inverse=True)
            probability=np.bincount(inv,weights=w,minlength=len(states))/w.sum()
            change=np.zeros(len(a))
            for state,p in zip(states,probability):
                modified=a.copy();modified[:,cols]=state
                change+=p*(fi_predict_model(m,answers=modified)!=labels)
            meta=dict(family=family,wave=y,question=name,kind=kind)
            pieces['questions'].append(dict(**meta,expected_changed=float(w@change/w.sum()),states=len(states)))
            for c in range(k):
                ok=labels==c
                pieces['question_clusters'].append(dict(**meta,cluster=c,expected_loss=float(w[ok]@change[ok]/w[ok].sum())))
    return {name:pd.DataFrame(rows) for name,rows in pieces.items()}

def fi_check_and_save(tables,family):
    keys=dict(separation=['family','wave','stage','feature'],cluster_separation=['family','wave','stage','cluster','feature'],
        omissions=['family','wave','kind','feature'],omission_clusters=['family','wave','kind','feature','cluster'],
        questions=['family','wave','kind','question'],question_clusters=['family','wave','kind','question','cluster'])
    numeric=dict(separation=['TSS','BSS','WSS','separation_share','eta_squared'],
        cluster_separation=['BSS','within_cluster_separation_share'],omissions=['changed'],
        omission_clusters=['recovery','Jaccard'],questions=['expected_changed','states'],question_clusters=['expected_loss'])
    folder=FI_ASSETS.parent/'results';folder.mkdir(exist_ok=True)
    for name,frame in tables.items():
        expected=pd.read_csv(FI_ASSETS/f'feature_importance_expected_{name}.csv',float_precision='round_trip')
        expected=expected[expected.family.eq(family)]
        left=frame.sort_values(keys[name]).reset_index(drop=True)
        right=expected.sort_values(keys[name]).reset_index(drop=True)
        pd.testing.assert_frame_equal(left[keys[name]],right[keys[name]],check_dtype=False)
        np.testing.assert_allclose(left[numeric[name]],right[numeric[name]],atol=3e-12,rtol=1e-10)
        frame.to_csv(folder/f'feature_importance_{family}_{name}.csv',index=False)
    print('Verified and saved',family,'feature-importance tables; no models refitted.')

In [8]:
FI_ASSETS=ASSETS
fi_manifest=json.loads((FI_ASSETS/'feature_importance_manifest.json').read_text())
for filename,expected in fi_manifest['files_sha256'].items():
    assert hashlib.sha256((FI_ASSETS/filename).read_bytes()).hexdigest()==expected
fi_models={}
for y in [2021,2023,2025]:
    a=evidence[f'native_{y}_answers'];X=fi_full_geometry(a)
    fi_models['full',y]=dict(family='full',wave=y,answers=a,weights=evidence[f'native_{y}_mass'],X=X,
        centres=evidence[f'{y}_centres'],groups=evidence[f'{y}_centre_to_cluster'],labels=evidence[f'native_{y}_labels'],k=5)
fi_tables=fi_inspect(fi_models)
fi_check_and_save(fi_tables,'full')
fi_s=fi_tables['separation'];fi_o=fi_tables['omissions']
fi_display=fi_s[fi_s.wave.eq(2025)].merge(fi_o[fi_o.wave.eq(2025)&fi_o.kind.eq('coordinate')],on=['family','wave','feature'])
fi_display['separation_pct']=100*fi_display.separation_share;fi_display['changed_pct']=100*fi_display.changed
display(fi_display[['feature','separation_pct','changed_pct']].sort_values('separation_pct',ascending=False).round(2))
display(fi_o[fi_o.feature.eq('Overall worry mean')][['wave','changed']].set_index('wave').mul(100).round(2))
display(fi_o[fi_o.wave.eq(2025)&fi_o.kind.eq('block')][['feature','changed']].set_index('feature').mul(100).round(2))
fi_q=fi_tables['questions']
display(fi_q[fi_q.kind.eq('original worry question')].pivot(index='question',columns='wave',values='expected_changed').mul(100).round(2))

Verified and saved full feature-importance tables; no models refitted.


,feature,separation_pct,changed_pct
6,Overall worry mean,20.87,10.86
5,mental health (relative worry),17.51,8.56
1,water (relative worry),14.80,8.35
4,traffic (relative worry),8.05,4.78
0,food (relative worry),7.79,5.01
2,crime (relative worry),4.03,4.07
18,Known-person harm: traffic,3.79,3.07
16,Known-person harm: crime,3.75,2.97
19,Known-person harm: mental health,3.35,2.98
20,Known-person harm: work,3.10,2.72


,changed
wave,
2021,9.82
2023,10.92
2025,10.86


,changed
feature,
All relative worries,35.73
All worry coordinates,53.91
All personal harm,2.09
All known-person harm,10.20
All harm,12.34


wave,2021,2023,2025
question,,,
crime worry,19.68,19.56,18.79
food worry,10.75,12.27,12.42
mental health worry,25.51,23.24,24.42
severe weather worry,19.22,20.42,17.29
traffic worry,22.88,22.63,23.24
water worry,24.14,23.56,22.50


The mean is the strongest single encoded coordinate in 2025: 20.87% of separation and 10.86% reassigned on omission. Mental-health and water deviations follow by omission. Known-person harm matters as a block: omitting its seven coordinates changes 10.20%, versus 2.09% for personal harm. The following check repeats mean omission on all 300 saved confirmation models, using their saved centres and fixed maps.

In [9]:
# Inspect every saved confirmation fit on the same native development population.
# Only fitted centres and their fixed maps are exported; no respondent rows or
# sampled respondent indices are needed by this check.
with np.load(FI_ASSETS/'feature_importance_refit_models.npz',allow_pickle=False) as pack:
    fi_refits={key:pack[key] for key in pack.files}
assert fi_refits['centres'].shape==(3,2,50,7,22)
fi_rows=[]
for yi,y in enumerate(fi_refits['years']):
    m=fi_models['full',int(y)];X=m['X'];w=m['weights'];keep=np.arange(22)!=6
    for ti,test in enumerate(fi_refits['tests']):
        for repeat in range(50):
            C=fi_refits['centres'][yi,ti,repeat];g=fi_refits['maps'][yi,ti,repeat]
            ref=g[cdist(fi_full_geometry(evidence['reference_answers']),C,'sqeuclidean').argmin(1)]
            np.testing.assert_array_equal(ref,evidence[f'{y}_{test}_predictions'][repeat])
            before=g[cdist(X,C,'sqeuclidean').argmin(1)]
            after=g[cdist(X[:,keep],C[:,keep],'sqeuclidean').argmin(1)]
            fi_rows.append(dict(family='full',wave=int(y),test=str(test),repeat=repeat,changed=float(w[before!=after].sum())))
fi_refit_table=pd.DataFrame(fi_rows)
fi_expected=pd.read_csv(FI_ASSETS/'feature_importance_expected_mean_refits.csv',float_precision='round_trip')
sort=['family','wave','test','repeat']
np.testing.assert_allclose(fi_refit_table.sort_values(sort).changed,fi_expected.sort_values(sort).changed,atol=3e-12)
fi_refit_table.to_csv(FI_ASSETS.parent/'results/feature_importance_full_mean_refits.csv',index=False)
display(fi_refit_table.groupby(['wave','test']).changed.agg(['min','median','max']).mul(100).round(2))

min  median    max
wave test                             
2021 countries     9.59   10.26  10.65
     respondents   9.70   10.19  10.56
2023 countries    10.55   10.81  11.04
     respondents  10.69   10.81  10.97
2025 countries    10.40   10.76  11.03
     respondents  10.51   10.82  11.01